# 04. Causal mask: запрет смотреть в будущее

Лекция: [Causal mask](../../notes/interview-prep/08b-causal-mask.md).

Цель: построить маску, применить её до softmax и проверить причинность экспериментом. Notebook независим от предыдущего. Сегодня выполняй до **остановки A**, сохрани и пришли на проверку. B и C — следующие небольшие этапы. Маску реализуем для self-attention без padding и dropout.

## A1. Кто кого может видеть?

Строка — Query получателя, столбец — Key источника. Источник j разрешён, если j <= i. Диагональ разрешена: текущий входной токен известен, а предсказывать будем следующий.

Для трёх позиций разрешены связи:

```text
         Key 0  Key 1  Key 2
Query 0    1      0      0
Query 1    1      1      0
Query 2    1      1      1
```

Здесь используем **blocked_mask: True = запретить**. Это наше соглашение для masked_fill, а не универсальное соглашение всех attention API.

In [1]:
import math
from cmath import inf

import torch
from torch import nn

torch.manual_seed(42)
torch.set_printoptions(precision=3, sci_mode=False)
T = 3

query_positions = torch.arange(T)[:, None]  # (T, 1)
key_positions = torch.arange(T)[None, :]    # (1, T)
print('Query positions:', query_positions)
print('Key positions:', key_positions)

Query positions: tensor([[0],
        [1],
        [2]])
Key positions: tensor([[0, 1, 2]])


## A2. Построй булеву маску

Broadcasting сравнит каждую позицию Query с каждой позицией Key. В ячейке [i,j] нужно True, когда источник j находится в будущем относительно получателя i.

Подсказка: сравни key_positions и query_positions оператором `>` или `<`. Не копируй готовую таблицу: правило должно работать для любой T.

In [2]:

blocked_mask = key_positions > query_positions

assert isinstance(blocked_mask, torch.Tensor), 'Замени ... сравнением позиций'
assert blocked_mask.dtype == torch.bool
assert blocked_mask.shape == (T, T)
expected_mask = torch.tensor([
    [False, True, True],
    [False, False, True],
    [False, False, False],
])
assert torch.equal(blocked_mask, expected_mask)
print('True = запрещено:')
print(blocked_mask)

True = запрещено:
tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])


## A3. Почему не просто заменить scores нулями?

Softmax использует экспоненты. exp(0)=1, поэтому нулевой score не запрещает позицию. Нужен -inf: exp(-inf)=0.

Рассмотрим первый токен: ему разрешён только первый источник. Следующая демонстрация уже готова.

In [3]:
zeroed_scores = torch.tensor([2.0, 0.0, 0.0])
masked_example = torch.tensor([2.0, float('-inf'), float('-inf')])
print('Замена на 0:', torch.softmax(zeroed_scores, dim=-1))
print('Замена на -inf:', torch.softmax(masked_example, dim=-1))
torch.testing.assert_close(
    torch.softmax(masked_example, dim=-1),
    torch.tensor([1.0, 0.0, 0.0]),
)

Замена на 0: tensor([0.787, 0.107, 0.107])
Замена на -inf: tensor([1., 0., 0.])


### Остановка A — сохрани и пришли на проверку

1. Что означает blocked_mask[0,2] == True?
2. Почему blocked_mask[2,0] == False?
3. Почему разрешена диагональ? Что предсказывает модель с текущей позиции?
4. Почему 0 не подходит для запрещённого score?

**Мои ответы:**

1. Query первой позиции не может использовать Key/Value третьей позиции: третий токен находится в будущем. Индексы маски — [позиция получателя, позиция источника]; True означает запрет.
2. Третья позиция может использовать первую: источник находится в прошлом. Для [2,0] условие запрета j > i ложно.
3. Диагональ соответствует вниманию к собственной позиции. Она разрешена, поскольку текущий входной токен уже известен, а модель с этой позиции предсказывает следующий токен. Например, из позиции «кот» предсказываем «съел»: читать «кот» можно, а будущий «съел» — подсматривание.
4. exp(0)=1, но это ещё не результат softmax: затем значение делится на сумму экспонент строки. Поэтому нулевой score получает ненулевой вес. В нашем примере softmax([2,0,0]) примерно равен [0.787,0.107,0.107]. Замена запрещённых scores на -inf даёт нулевые веса, если в строке остаётся хотя бы один разрешённый конечный score.

Не обязательно делать всё за один раз. К следующему этапу перейдём после разбора.

## B. Применяем маску к attention

Восстановим знакомые X, Q, K, V. Маска формы (T,T) будет общей для всех последовательностей batch формы (B,T,T). Она не смешивает batch.

In [4]:
token_ids = torch.tensor([[1, 2, 3], [3, 2, 1]])
B, T = token_ids.shape
D = 4

token_embedding = nn.Embedding(4, D)
position_embedding = nn.Embedding(16, D)
query_projection = nn.Linear(D, D, bias=False)
key_projection = nn.Linear(D, D, bias=False)
value_projection = nn.Linear(D, D, bias=False)

X = token_embedding(token_ids) + position_embedding(torch.arange(T))
Q = query_projection(X)
K = key_projection(X)
V = value_projection(X)
scaled_scores = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.shape[-1])

`tensor.masked_fill(mask, value)` возвращает тензор, в котором элементы под True заменены на value. Применяй к scaled_scores, затем softmax, затем смешивание Values.

In [5]:
masked_scores = scaled_scores.masked_fill(blocked_mask, float('-inf'))
attention_weights = torch.softmax(masked_scores, dim=-1)
context = attention_weights @ V

assert isinstance(masked_scores, torch.Tensor)
assert isinstance(attention_weights, torch.Tensor)
assert isinstance(context, torch.Tensor)
assert context.shape == (B, T, D)
expanded_mask = blocked_mask.unsqueeze(0).expand(B, -1, -1)
assert torch.isneginf(masked_scores[expanded_mask]).all()
torch.testing.assert_close(masked_scores[~expanded_mask], scaled_scores[~expanded_mask])
assert torch.all(attention_weights[expanded_mask] == 0)
torch.testing.assert_close(attention_weights.sum(-1), torch.ones(B, T))
torch.testing.assert_close(context[:, 0], V[:, 0])
assert torch.isfinite(context).all()
print('Веса первой последовательности:')
print(attention_weights[0])
print('Context:', context.shape)

Веса первой последовательности:
tensor([[1.000, 0.000, 0.000],
        [0.592, 0.408, 0.000],
        [0.101, 0.382, 0.517]], grad_fn=<SelectBackward0>)
Context: torch.Size([2, 3, 4])


### Остановка B

1. Почему context[:,0] совпал с V[:,0]?
2. Сколько источников доступно Query позиции 1?
3. Почему маска применяется до softmax? Что произойдёт с суммой строки, если просто занулить веса после softmax без повторной нормировки?

**Мои ответы:**

1. Первая позиция может использовать только собственный Value. Её веса равны [1,0,0], поэтому context[:,0] = 1*V[:,0]. Это результат нашего attention без dropout и последующих преобразований.
2. Два источника: позиция 0 и сама позиция 1. Индексация начинается с нуля.
3. Маска до softmax исключает запрещённые источники из нормировки. Если после softmax просто занулить положительные веса будущих позиций, сумма строки станет меньше 1: например, [0.2,0.3,0.5] превратится в [0.2,0.3,0]. Это уже не нормированное распределение по источникам. Маскирование scores до softmax сразу распределяет единицу веса между разрешёнными позициями.

## C. Эксперимент: меняем будущее

Заполни функцию знакомыми операциями. Размер маски вычисляй из текущего входа, а устройство бери у X. При causal=False маска не применяется. Здесь вход X — ещё не контекстное представление: изменяем только последнюю позицию.

Используем те же слои, чтобы сравнение не смешивалось с изменением параметров. Упаковку в собственный nn.Module сделаем после проверки этого эксперимента.

In [6]:
def attend(x, causal=True):
    q = query_projection(x)
    k = key_projection(x)
    v = value_projection(x)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
    if causal:
        positions = torch.arange(x.shape[1], device=x.device)

        mask = positions[None, :] > positions[:, None]
        assert isinstance(mask, torch.Tensor), 'Создай маску из positions'
        scores = scores.masked_fill(mask, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ v


In [7]:
# Готовый тест. Изменяется только последняя позиция каждой последовательности.
# no_grad нужен для эксперимента без построения графа, а не для причинности.
with torch.no_grad():
    changed_X = X.clone()
    changed_X[:, -1] += torch.tensor([10.0, -7.0, 5.0, 3.0])
    before = attend(X, causal=True)
    after = attend(changed_X, causal=True)
    torch.testing.assert_close(before[:, :-1], after[:, :-1])
    print('Causal: прошлые позиции не изменились — проверка пройдена')
    print('Causal: изменение последней позиции:', (before[:, -1] - after[:, -1]).abs().max().item())
    unmasked_before = attend(X, causal=False)
    unmasked_after = attend(changed_X, causal=False)
    print('Без маски: изменение прошлых позиций:',
          (unmasked_before[:, :-1] - unmasked_after[:, :-1]).abs().max().item())

# Не утверждаем, что без маски изменение обязательно ненулевое для любых параметров.
# Causal attention гарантирует отсутствие зависимости от будущего в этом вычислении.

Causal: прошлые позиции не изменились — проверка пройдена
Causal: изменение последней позиции: 0.880083441734314
Без маски: изменение прошлых позиций: 2.6087822914123535


### Остановка C

1. Почему изменение последнего X не повлияло на предыдущие выходы с causal mask?
2. Что этот тест проверяет сверх нулей в матрице внимания?
3. Почему маска не мешает вычислить все строки attention одновременно во время обучения?

**Мои ответы:**

1. Предыдущие позиции не используют последний, будущий для них источник. Их Queries и разрешённые им Keys/Values не изменились, поэтому выходы совпали. Мы изменяли только последний входной вектор X, а параметры слоёв оставили прежними.
2. Проверяется поведение всей функции attention: изменение будущего не должно влиять на прошлые выходы. Проверка нулей в attention_weights проверяет промежуточный результат, а этот эксперимент — отсутствие зависимости конечного context от будущего. В сохранённом запуске проверка совпадения прошлых выходов с маской прошла; без маски максимальное абсолютное изменение прошлых выходов было примерно 2.609.
3. Во время обучения весь вход уже известен. Q/K/V всех позиций вычисляются из входа слоя, затем матричными операциями считаются scores, маскирование, softmax и context. Для строки позиции 2 не нужен готовый context позиции 1: строки не зависят от выходов друг друга внутри этого attention. Маска ограничивает связи, а не требует вычислять строки по очереди. При генерации следующий входной токен ещё неизвестен, поэтому последовательны шаги генерации.

Ответы уточнены вместе с ментором после проверки выполненного кода. Практика causal mask завершена по сохранённым результатам. Следующее задание — отдельный notebook: упаковка attention в собственный nn.Module.

Перед финальной проверкой: Restart Kernel → Run All → сохранить outputs. Намеренно не маскируем всю строку: softmax строки из одних -inf даёт NaN. Padding рассмотрим отдельно.